In [ ]:
# Cell 1: Install Libraries
# Installs necessary packages for the application.

print("⏳ Installing libraries...")

# For AI Interaction: google-generativeai
# For UI: gradio
# For Web Scraping: requests, beautifulsoup4
# For Markdown Processing: markdown, markdownify (for TXT export)
# For File Processing: PyPDF2 (PDFs), python-docx (Word Docs)
# For Scorecard Plotting: matplotlib
# For General Utilities: importlib-metadata (used once in imports cell)
!pip install -q google-generativeai gradio requests beautifulsoup4 markdown PyPDF2 python-docx markdownify matplotlib importlib-metadata

print("✅ Required libraries installed.")

In [ ]:
# Cell 2: Import Libraries
# Imports all necessary modules for the application.

print("⏳ Importing libraries...")

# Core AI and UI
import google.generativeai as genai
import gradio as gr

# Web Scraping & Parsing
import requests
from bs4 import BeautifulSoup
import re # For regular expressions (text cleaning, parsing)

# File Processing
import os # For os.path.basename AND os.environ for local API key
import traceback # For detailed error logging
try:
    import PyPDF2
    print("  ✅ PyPDF2 imported.")
except ImportError:
    print("  ⚠️ PyPDF2 not found. PDF processing will be disabled.")
    PyPDF2 = None
try:
    import docx
    print("  ✅ python-docx imported.")
except ImportError:
    print("  ⚠️ python-docx not found. DOCX processing will be disabled.")
    docx = None

# Output Formatting & Export
import markdown # Potentially used by Gradio Markdown component
try:
    import markdownify
    print("  ✅ Markdownify imported.")
except ImportError:
    print("  ⚠️ Markdownify not found. TXT download will use basic fallback.")
    markdownify = None

# Secrets Management (Multi-environment)
# Kaggle specific
try:
    from kaggle_secrets import UserSecretsClient
    print("  ✅ Kaggle secrets client imported (will be used if in Kaggle).")
except ImportError:
    print("  ℹ️ Kaggle secrets client not found (expected if not in Kaggle).")
    UserSecretsClient = None

# Colab specific
try:
    from google.colab import userdata
    print("  ✅ Google Colab userdata imported (will be used if in Colab).")
except ImportError:
    print("  ℹ️ Google Colab userdata not found (expected if not in Colab).")
    userdata = None

# For checking library versions (optional debug)
try:
    import importlib.metadata
except ImportError:
    importlib = None # Gracefully handle if not available

# General Utilities
import tempfile # For creating temporary download files

print("✅ Libraries imported.")

In [ ]:
# Cell 3: API Configuration and AI Model Initialization
# Sets up the connection to the AI model using the API key.
# Checks for the key in Kaggle Secrets -> Colab Secrets -> Environment Variable.

# Import necessary modules if they aren't already guaranteed to be in the global scope
# (e.g., if running cells out of order or as separate scripts)
import os
import google.generativeai as genai

# Ensure UserSecretsClient and userdata are potentially defined from Cell 2
# Handle cases where they might not exist (e.g., running cell standalone)
try:
    # Check if UserSecretsClient was successfully imported in cell 2
    if 'UserSecretsClient' not in globals():
        # Attempt import again if needed, or set to None
        try:
            from kaggle_secrets import UserSecretsClient
        except ImportError:
            UserSecretsClient = None
except NameError:
     UserSecretsClient = None

try:
     # Check if userdata was successfully imported in cell 2
    if 'userdata' not in globals():
        # Attempt import again if needed, or set to None
        try:
            from google.colab import userdata
        except ImportError:
            userdata = None
except NameError:
    userdata = None


print("⏳ Configuring AI API and initializing model...")

# --- Configuration ---
API_KEY_SECRET_NAME = "GOOGLE_API_KEY" # Name of the secret in Kaggle/Colab/Env Var
DEFAULT_MODEL_NAME = 'gemini-2.5-pro-exp-03-25' # Or 'gemini-1.5-flash-latest'
print(f"  ℹ️ Target AI Model: {DEFAULT_MODEL_NAME}")
print(f"  ℹ️ API Key Name: {API_KEY_SECRET_NAME}")

api_key = None
api_key_configured = False
model = None # Initialize model to None
key_source = "Not Found"

# --- Prioritized API Key Retrieval ---

# 1. Check Kaggle Secrets
if UserSecretsClient:
    try:
        print("  ⏳ Checking Kaggle Secrets...")
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret(API_KEY_SECRET_NAME)
        if api_key:
            print("  🔑 API Key found in Kaggle Secrets.")
            key_source = "Kaggle Secrets"
        else:
            print(f"  ℹ️ API Key '{API_KEY_SECRET_NAME}' not found in Kaggle Secrets.")
    except Exception as e:
        # Catch potential errors like the secret not existing
        print(f"  ⚠️ Warning: Error accessing Kaggle Secrets: {e}")
        api_key = None # Ensure key is None if error occurs

# 2. Check Colab Secrets (if not found in Kaggle)
if not api_key and userdata:
    try:
        print("  ⏳ Checking Google Colab Secrets...")
        api_key = userdata.get(API_KEY_SECRET_NAME)
        if api_key:
            print("  🔑 API Key found in Google Colab Secrets.")
            key_source = "Colab Secrets"
        else:
            print(f"  ℹ️ API Key '{API_KEY_SECRET_NAME}' not found in Colab Secrets.")
    except Exception as e:
        # Catch potential errors like the secret not existing
        print(f"  ⚠️ Warning: Error accessing Colab Secrets: {e}")
        api_key = None # Ensure key is None if error occurs

# 3. Check Environment Variable (if not found in Kaggle or Colab)
if not api_key:
    try:
        print(f"  ⏳ Checking Environment Variable ('{API_KEY_SECRET_NAME}')...")
        api_key = os.environ.get(API_KEY_SECRET_NAME)
        if api_key:
            print("  🔑 API Key found in Environment Variable.")
            key_source = "Environment Variable"
        else:
            print(f"  ℹ️ API Key not found as Environment Variable '{API_KEY_SECRET_NAME}'.")
    except Exception as e:
        # This shouldn't typically fail for os.environ.get, but include for safety
        print(f"  ⚠️ Warning: Error accessing Environment Variable: {e}")
        api_key = None

# --- Configure GenerativeAI ---
if api_key:
    try:
        genai.configure(api_key=api_key)
        print(f"  ✅ Google API Key configured successfully via {key_source}.")
        api_key_configured = True
    except Exception as e:
        print(f"  ❌ ERROR: Failed to configure genai with the retrieved API key: {e}")
        api_key_configured = False
else:
    print(f"  ❌ ERROR: API Key '{API_KEY_SECRET_NAME}' not found in Kaggle Secrets, Colab Secrets, or Environment Variables.")
    print(f"  ⚠️ Please ensure the key is available in one of these locations and named correctly.")

# --- Initialize Model ---
if api_key_configured:
    try:
        print(f"  ⏳ Initializing AI model: {DEFAULT_MODEL_NAME}...")
        # Define safety settings to block harmful content
        # You can adjust thresholds: BLOCK_NONE, BLOCK_LOW_AND_ABOVE, BLOCK_MEDIUM_AND_ABOVE, BLOCK_ONLY_HIGH
        safety_settings = [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
        ]
        model = genai.GenerativeModel(DEFAULT_MODEL_NAME, safety_settings=safety_settings)
        # Optional quick check to confirm model access (uncomment to use)
        # print("  🧪 Performing quick model test...")
        # test_response = model.generate_content("test", generation_config=genai.types.GenerationConfig(max_output_tokens=5))
        # print(f"  🧪 Model test response snippet: {test_response.text[:50]}...")
        print(f"  🤖 AI model '{model.model_name}' initialized successfully.")
    except Exception as e:
        print(f"  ❌ ERROR: Failed to initialize AI model '{DEFAULT_MODEL_NAME}': {e}")
        print(f"  ⚠️ Check if the model name is correct, available in your region, and if the API key has permissions for this model.")
        model = None
else:
    print("  ⚠️ Skipping AI model initialization because API key is not configured.")

# --- Final Status ---
if api_key_configured and model:
    print("✅ API Configuration and Model Initialization successful.")
else:
    print("❌ API Configuration or Model Initialization failed. Generation functions will not work.")

In [ ]:
# Cell 4: Application Constants

print("⏳ Defining application constants...")

INDUSTRY_CHOICES = [
    "Manufacturing", "Logistics", "Healthcare", "Finance", "Retail",
    "Technology (IT)", "Human Resources (HR)", "Education", "Marketing",
    "Professional Services (Consulting, Legal, Accounting)", "Hospitality (Restaurants, Hotels)",
    "Construction & Real Estate", "Other"
]

TONE_CHOICES = [
    "Informative & Actionable", "Formal & Authoritative", "Casual & Engaging",
    "Persuasive & Benefit-Driven", "Technical & Detailed", "Blog Post Style",
    "Case Study Format", "News/Update Style"
]

# Focus primarily on SMBs, but allow other options
TARGET_BUSINESS_CHOICES = [
    "Small and Medium Businesses (SMB)", "Specific SMB Sector (Defined Below)",
    "Enterprise", "Startups", "Non-profit", "General Audience"
]

print("✅ Application constants defined.")

In [ ]:
# Cell 5: Scraping and File Processing Function
# Defines the function to extract text from URLs and Files with enhanced logging.

print("⏳ Defining scraping and file processing function...")

def scrape_urls_and_files(url_list_str, uploaded_files=None):
    """
    Scrapes text from URLs (requests/BS) and extracts text from files (PDF, DOCX).
    Includes enhanced logging style.

    Args:
        url_list_str (str): URLs separated by newlines.
        uploaded_files (list): List of Gradio file objects.

    Returns:
        tuple: (str, str, list) - A tuple containing:
            - combined_text (str): Concatenated text from all sources.
            - final_status (str): Detailed status message for UI.
            - successful_sources_list (list): List of successfully processed source names.
    """
    # Ensure necessary libraries are available in this scope
    global PyPDF2, docx # Access libraries imported in Cell 2

    print(f"\n⏳ [Processing Sources] Function called...", flush=True)
    urls = [url.strip() for url in url_list_str.strip().split('\n') if url.strip()]
    combined_text = ""
    status_messages = []
    successful_sources_list = []
    successful_scrapes = 0
    successful_files = 0
    processed_sources_count = 0

    # --- Scrape URLs ---
    if urls:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        print(f"  ⏳ Starting URL scraping for {len(urls)} URLs...", flush=True)
        processed_sources_count += len(urls)
        for i, url in enumerate(urls):
            source_name = url
            print(f"    ⏳ Processing URL {i+1}/{len(urls)}: {url}", flush=True)
            try:
                response = requests.get(url, headers=headers, timeout=20)
                print(f"      -> Status Code: {response.status_code}", flush=True)
                response.raise_for_status() # Raise HTTPError for bad status (4xx or 5xx)

                print(f"      -> Parsing HTML...", flush=True)
                soup = BeautifulSoup(response.content, 'html.parser')
                # Remove common non-content tags
                for script_or_style in soup(["script", "style", "nav", "footer", "aside", "header", "form"]):
                    if script_or_style: script_or_style.decompose()

                print(f"      -> Extracting text...", flush=True)
                text_parts = []
                # Try common content containers first
                main_content = soup.find('article') or soup.find('main') or soup.find('div', id=re.compile(r'content|main|body', re.I)) or soup.find('div', class_=re.compile(r'content|main|body|post|entry|article', re.I))
                if main_content:
                     print(f"      -> Found main content container. Extracting paragraphs, headings, lists.", flush=True)
                     for element in main_content.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li']):
                         text_parts.append(element.get_text(separator=' ', strip=True))
                else:
                     # Fallback if specific containers aren't found
                     print(f"      ⚠️ Warning: No main content tags found. Falling back to all <p> tags.", flush=True)
                     for element in soup.find_all('p'):
                          text_parts.append(element.get_text(separator=' ', strip=True))

                page_text_from_parts = "\n".join(filter(None, text_parts))
                # Fallback to body if extracted text is still minimal
                if not page_text_from_parts or len(page_text_from_parts) < 150:
                    print(f"      ⚠️ Warning: Minimal text in tags/paragraphs. Trying full body text.", flush=True)
                    body_text = soup.body.get_text(separator='\n', strip=True) if soup.body else "" # Use newline separator for body
                    page_text = body_text
                else:
                    page_text = page_text_from_parts

                # Basic cleaning (more aggressive whitespace removal)
                page_text = re.sub(r'[ \t]+', ' ', page_text) if page_text else "" # Replace multiple spaces/tabs with single space
                page_text = re.sub(r'\n\s*\n', '\n\n', page_text) # Replace multiple newlines (with optional spaces) with double newline
                page_text = page_text.strip()

                if page_text:
                    combined_text += f"\n\n--- Content from URL: {source_name} ---\n\n{page_text}"
                    status_msg_ui = f"✅ Successfully processed: {source_name}"
                    status_messages.append(status_msg_ui)
                    successful_sources_list.append(source_name)
                    successful_scrapes += 1
                    print(f"      -> Status: ✅ OK ({len(page_text)} chars extracted)", flush=True)
                else:
                    status_msg_ui = f"⚠️ No significant text content found at: {source_name}"
                    status_messages.append(status_msg_ui)
                    print(f"      -> Status: ⚠️ Warning (No significant text found)", flush=True)

            except requests.exceptions.Timeout:
                 err_msg_ui = f"❌ Timeout Error accessing {source_name}"; status_messages.append(err_msg_ui); print(f"      -> Status: ❌ ERROR (Timeout)", flush=True)
            except requests.exceptions.HTTPError as e:
                 err_msg_ui = f"❌ HTTP Error accessing {source_name}: Status {e.response.status_code}"; status_messages.append(err_msg_ui); print(f"      -> Status: ❌ ERROR (HTTP {e.response.status_code})", flush=True)
            except requests.exceptions.RequestException as e:
                 err_msg_ui = f"❌ Network Error accessing {source_name}: {type(e).__name__}"; status_messages.append(err_msg_ui); print(f"      -> Status: ❌ ERROR (Network: {type(e).__name__})", flush=True)
            except Exception as e:
                 err_msg_ui = f"❌ Error processing {source_name}: {type(e).__name__} - {str(e)[:100]}"; status_messages.append(err_msg_ui); print(f"      -> Status: ❌ ERROR (Processing: {type(e).__name__})", flush=True); traceback.print_exc()
        print(f"  ✅ URL scraping finished.", flush=True)

    # --- Process Uploaded Files ---
    if uploaded_files:
        print(f"  ⏳ Starting file processing for {len(uploaded_files)} files...", flush=True)
        processed_sources_count += len(uploaded_files)
        for i, file_obj in enumerate(uploaded_files):
            file_path = file_obj.name
            file_name = os.path.basename(file_path)
            source_name = f"File: {file_name}"
            print(f"    ⏳ Processing File {i+1}/{len(uploaded_files)}: {file_name}", flush=True)
            file_text = ""
            try:
                if file_path.lower().endswith(".pdf") and PyPDF2:
                    print(f"      -> Extracting text from PDF...", flush=True)
                    reader = PyPDF2.PdfReader(file_path); num_pages = len(reader.pages)
                    for page_num, page in enumerate(reader.pages): file_text += page.extract_text() + "\n"
                    print(f"      -> Extracted text from {num_pages} PDF pages.", flush=True)
                elif file_path.lower().endswith(".docx") and docx:
                    print(f"      -> Extracting text from DOCX...", flush=True)
                    document = docx.Document(file_path); num_paras = len(document.paragraphs)
                    for para in document.paragraphs: file_text += para.text + "\n"
                    print(f"      -> Extracted text from {num_paras} DOCX paragraphs.", flush=True)
                else:
                    status_msg_ui = f"⚠️ Skipped unsupported/unreadable file: {file_name}"; status_messages.append(status_msg_ui); print(f"      -> Status: ⚠️ Warning (Skipped unsupported file type or missing library)", flush=True); continue

                # Clean extracted file text
                file_text = re.sub(r'\s{2,}', ' ', file_text).strip() # Replace multiple whitespace chars with single space
                file_text = re.sub(r'\n{3,}', '\n\n', file_text).strip() # Consolidate multiple newlines

                if file_text:
                    combined_text += f"\n\n--- Content from {source_name} ---\n\n{file_text}"
                    status_msg_ui = f"✅ Successfully processed: {source_name}"; status_messages.append(status_msg_ui); successful_sources_list.append(source_name); successful_files += 1; print(f"      -> Status: ✅ OK ({len(file_text)} chars extracted)", flush=True)
                else:
                    status_msg_ui = f"⚠️ No text content extracted from: {source_name}"; status_messages.append(status_msg_ui); print(f"      -> Status: ⚠️ Warning (No text extracted)", flush=True)

            except ImportError as e:
                 err_msg_ui = f"❌ Missing library for {file_name}. Install required libraries."; status_messages.append(err_msg_ui); print(f"      -> Status: ❌ ERROR (Missing library: {e})", flush=True)
            except Exception as e:
                 err_msg_ui = f"❌ Error processing {source_name}: {type(e).__name__} - {str(e)[:100]}"; status_messages.append(err_msg_ui); print(f"      -> Status: ❌ ERROR (Processing: {type(e).__name__})", flush=True); traceback.print_exc()
        print(f"  ✅ File processing finished.", flush=True)


    # --- Final Status ---
    if not urls and not uploaded_files:
         final_status = "ℹ️ Please enter URLs or upload files to process."
    else:
        total_successful = successful_scrapes + successful_files
        status_details = "\n".join(status_messages) if status_messages else "No specific status messages."
        final_status = (f"✅ Processing finished. Attempted {processed_sources_count} sources. "
                        f"Successfully processed {total_successful} sources.\n\n"
                        f"{status_details}")

    print(f"📊 [Processing Sources] Final Summary:", flush=True)
    print(f"  - Attempted: {processed_sources_count}, Succeeded: {total_successful}", flush=True)
    print(f"  - Successful sources list: {successful_sources_list}", flush=True)
    print(f"✅ [Processing Sources] Function finished.", flush=True)

    # Return 3 values
    return (combined_text.strip(), final_status, successful_sources_list)

print("✅ Scraping and file processing function defined.")

In [ ]:
# Cell 6: AI Outline Generation Function Definition
# Defines the function that interacts with the AI model to generate an article outline.

print("⏳ Defining AI outline generation function...")

def generate_outline_with_gemini(context, user_prompt, industry, target_business, specific_smb_sector=None):
    """
    Generates an article outline using the AI model based on provided context and parameters,
    with a strong focus on SMB benefits.

    Args:
        context (str): The scraped/extracted text content.
        user_prompt (str): The user's specific request for the article topic.
        industry (str): The target industry.
        target_business (str): The target business size/type.
        specific_smb_sector (str, optional): Specific SMB sector if target_business indicates it.

    Returns:
        str: The generated outline in Markdown format, or an error message.
    """
    global model # Access the initialized model from Cell 3
    print(f"\n⏳ [AI Outline Generation] Function called...", flush=True)

    # --- Input Validation ---
    if not model:
        print("❌ ERROR: AI model not initialized.")
        return "❌ Error: AI model not initialized. Please check API key configuration in Cell 3."
    if not user_prompt or user_prompt.strip() == "":
        print("❌ ERROR: User prompt empty.")
        return "❌ Error: Please provide an Article Prompt."
    if not context or context.strip() == "":
        print("❌ ERROR: Context empty.")
        return "❌ Error: No text content from sources found. Please process sources in Tab 1."

    # Determine the specific SMB focus
    smb_focus_description = "Small and Medium Businesses (SMBs) in general"
    if target_business == "Specific SMB Sector (Defined Below)" and specific_smb_sector and specific_smb_sector.strip():
        smb_focus_description = f"SMBs specifically in the '{specific_smb_sector.strip()}' sector"
    elif target_business == "Small and Medium Businesses (SMB)":
         smb_focus_description = "Small and Medium Businesses (SMBs) in general"
    else:
        smb_focus_description = f"the target audience: {target_business}" # Fallback if not SMB focused

    print(f"  📝 Parameters: Industry='{industry}', Target='{target_business}', SMB Focus='{smb_focus_description}'", flush=True)

    # --- Construct Enhanced Prompt for Outline ---
    print("  ⏳ Constructing outline prompt for AI...", flush=True)
    prompt = f"""
    You are an expert content strategist tasked with creating a detailed article outline based on the provided context and user request. The primary goal is to structure an article that is highly relevant and beneficial for {smb_focus_description}.

    **Provided Context:**
    --- CONTEXT START ---
    {context}
    --- CONTEXT END ---

    **User Request:** "{user_prompt.strip()}"

    **Outline Requirements:**

    1.  **Analyze Context:** Thoroughly analyze the provided context to identify key themes, insights, data points, and arguments relevant to the user request and the target audience ({smb_focus_description}).
    2.  **Focus on SMB Benefits:** The entire outline structure must be geared towards demonstrating value and providing actionable insights specifically for {smb_focus_description} within the {industry} industry. Think about their common challenges, needs, and goals.
    3.  **Logical Flow:** Create a clear and logical flow for the article, starting with an engaging hook and progressing through key points to a strong conclusion and a specific SMB example.
    4.  **Output Format:** Generate the outline strictly in Markdown format. Use H2 (##) for main section headings and H3 (###) for subheadings/key points within sections. Do NOT include colons (:) in headings.
    5.  **Outline Components:** For each major section (H2), include:
        * **Subheadings (###):** Break down the section into specific topics or arguments using H3 headings.
        * **Keywords:** List 3-5 relevant keywords or key phrases under each H3 subheading that should be covered in that part of the article. Prefix keywords with "- Keyword: ".
        * **Brief Description:** Optionally, add a very brief (1 sentence) description under the H3 heading explaining the focus of that sub-point.
    6.  **Key Sections to Include:**
        * **## Engaging Title:** Suggest 1-2 compelling title options (without colons).
        * **## Introduction:** Outline the hook, the core problem/opportunity for SMBs, and the article's main takeaway.
        * **## [Main Body Section(s)]:** Create 2-4 main body sections (##) based on the context and user request, each focusing on a distinct aspect relevant to SMBs. Use H3 subheadings extensively within these.
        * **## Specific SMB Example:** Outline a concrete, practical example of how an SMB in the '{specific_smb_sector.strip() if specific_smb_sector else industry}' sector could apply the article's key takeaways. Be specific about the type of SMB and the application.
        * **## Conclusion:** Outline the key summary points and a final call to action or concluding thought for the SMB reader.

    **Example H3 Structure:**
    ### Subheading Title Here
    - Keyword: Relevant Term 1
    - Keyword: Key Concept A
    - Keyword: Actionable Phrase

    Generate the detailed article outline now, following all instructions carefully. Ensure the focus remains squarely on providing value to {smb_focus_description}.
    """
    print("  ✅ Outline prompt constructed.", flush=True)

    # --- API Call ---
    try:
        print(f"  ⏳ Sending request to AI API for outline (Prompt length: {len(prompt)} chars)...", flush=True)
        # Use moderate temperature for creative outlining
        generation_config = genai.types.GenerationConfig(temperature=0.7)
        # Safety settings are defined with the model in Cell 3
        response = model.generate_content(prompt, generation_config=generation_config)
        print("  ⏳ Received outline response from AI API.", flush=True)

        # --- Process Response ---
        if response.parts:
             generated_outline = response.text
             print("    -> Status: ✅ OK (Outline content received)", flush=True)
             if not generated_outline or generated_outline.strip() == "":
                 print("    -> Status: ⚠️ Warning (AI returned empty text for outline)", flush=True)
                 return "⚠️ Warning: AI returned an empty outline. Please try regenerating."
             else:
                 print(f"    -> Generated outline length: {len(generated_outline)}", flush=True)
                 # Basic cleanup: Remove potential leading/trailing ```markdown tags if present
                 generated_outline = re.sub(r'^```markdown\s*', '', generated_outline.strip())
                 generated_outline = re.sub(r'\s*```$', '', generated_outline)
                 return generated_outline.strip()
        else:
             feedback = response.prompt_feedback
             block_reason = feedback.block_reason if feedback else "Unknown"
             safety_ratings_str = str(feedback.safety_ratings) if feedback else "N/A"
             print(f"    -> Status: ❌ ERROR (AI outline response empty/blocked: {block_reason}) - Safety Ratings: {safety_ratings_str}", flush=True)
             error_message = f"❌ Error: AI outline response was empty or blocked (Reason: {block_reason})."
             if feedback and feedback.safety_ratings:
                 error_message += f"\nSafety Concerns: {safety_ratings_str}"
             return error_message

    except Exception as e:
        print(f"  ❌ ERROR: Exception during AI API call for outline:", flush=True)
        traceback.print_exc()
        error_details = str(e)
        # Check for specific API errors if possible (e.g., quota, permissions)
        if "API key not valid" in error_details:
            return "❌ Error: Invalid API Key. Please check configuration in Cell 3."
        elif "quota" in error_details.lower():
             return "❌ Error: API Quota Exceeded. Please check your usage limits."
        return f"❌ Error during outline generation: {type(e).__name__} - {error_details}"
    finally:
        print(f"✅ [AI Outline Generation] Function finished.", flush=True)


print("✅ AI outline generation function defined.")

In [ ]:
# Cell 7: AI Article Generation Function Definition
# Defines the function that interacts with the AI model to generate the full article based on an outline.

print("⏳ Defining AI article generation function...")

def generate_article_with_gemini(context, article_outline, industry, tone, target_business, word_count, specific_smb_sector=None, extra_instructions=None):
    """
    Generates an article using the AI model based on provided context, outline, and parameters.
    Focuses on SMBs, readability, mixed formatting, and includes a specific example.

    Args:
        context (str): The scraped/extracted text content (used for factual grounding).
        article_outline (str): The detailed article outline (potentially edited by the user).
        industry (str): The target industry.
        tone (str): The desired tone of the article.
        target_business (str): The target business size/type.
        word_count (int): The approximate desired word count.
        specific_smb_sector (str, optional): Specific SMB sector if relevant.
        extra_instructions (str, optional): Additional user prompts for regeneration.

    Returns:
        str: The generated article in Markdown format, or an error message.
    """
    global model # Access the initialized model from Cell 3
    print(f"\n⏳ [AI Article Generation] Function called...", flush=True)

    # --- Input Validation ---
    if not model:
        print("❌ ERROR: AI model not initialized.")
        return "❌ Error: AI model not initialized. Please check API key configuration in Cell 3."
    if not article_outline or article_outline.strip() == "":
        print("❌ ERROR: Article outline is empty.")
        return "❌ Error: Article Outline is empty. Please generate or provide an outline in Tab 2."
    if not context or context.strip() == "":
        # Allow generation without context if outline is very detailed, but warn
        print("⚠️ WARNING: Context is empty. Generation will rely solely on the outline.")
        # return "❌ Error: No text content from sources found. Please process sources in Tab 1."
    try:
        word_count = int(word_count)
        if word_count <= 0: raise ValueError("Word count must be positive.")
    except (ValueError, TypeError):
        print(f"❌ ERROR: Invalid word count: {word_count}. Using default 500.")
        word_count = 500 # Default value

    # Determine the specific SMB focus
    smb_focus_description = "Small and Medium Businesses (SMBs)"
    if target_business == "Specific SMB Sector (Defined Below)" and specific_smb_sector and specific_smb_sector.strip():
        smb_focus_description = f"SMBs specifically in the '{specific_smb_sector.strip()}' sector"
    elif target_business != "Small and Medium Businesses (SMB)":
        smb_focus_description = f"the target audience: {target_business}" # Fallback

    print(f"  📝 Parameters: Industry='{industry}', Tone='{tone}', Target='{target_business}', WC={word_count}, SMB Focus='{smb_focus_description}'", flush=True)
    if extra_instructions: print(f"  📝 Extra Instructions: {extra_instructions}", flush=True)


    # --- Construct Enhanced Prompt for Article ---
    print("  ⏳ Constructing article prompt for AI...", flush=True)
    prompt = f"""
    You are an expert content writer specializing in creating highly practical and engaging articles for Small and Medium Businesses (SMBs). Your task is to write a complete article based *strictly* on the provided Article Outline and grounded in the Provided Context.

    **Provided Context (for factual grounding):**
    --- CONTEXT START ---
    {context if context else "No context provided. Rely solely on the outline."}
    --- CONTEXT END ---

    **Article Outline to Follow:**
    --- OUTLINE START ---
    {article_outline}
    --- OUTLINE END ---

    **Article Requirements:**

    1.  **Adhere Strictly to Outline:** Follow the structure, headings, subheadings, keywords, and core ideas defined in the Article Outline precisely. Do NOT deviate or introduce new major sections.
    2.  **Focus:** The article MUST be written specifically for {smb_focus_description} operating in the {industry} industry. Every point should be framed in terms of its relevance, benefit, or application to these businesses.
    3.  **Content Generation:**
        * Expand on the points and keywords listed in the outline, using the Provided Context *only* for factual information, data, or supporting details where relevant. Do not hallucinate information not present in the context or outline.
        * If context is missing, rely entirely on the outline's structure and keywords to generate logical and helpful content appropriate for the SMB audience.
        * Desired Tone: {tone}.
        * Approximate Word Count: {word_count} words. Aim for this length, but prioritize covering the outline points well.
    4.  **Readability & Style:**
        * Write in clear, simple, and concise language. Avoid jargon where possible, or explain it clearly if necessary.
        * Use a mix of well-structured paragraphs (3-5 sentences typically) and bullet points (for lists or key takeaways). **Prioritize clear explanatory paragraphs** and use bullets strategically for clarity, not as the primary format.
        * Ensure a smooth and logical flow between sections and paragraphs. Use transition words and phrases.
    5.  **Formatting:**
        * Output the entire article strictly in Markdown format.
        * Use Markdown H1 (#) for the main title (use one suggested in the outline or create a similar one, NO colon).
        * Use Markdown H2 (##), H3 (###), etc., exactly as specified in the outline for headings and subheadings (NO colons).
        * Use **bold** (Markdown **) for emphasis on key terms or takeaways, aligning with the keywords from the outline where appropriate.
    6.  **Specific SMB Example:** Ensure the dedicated section for the SMB example is detailed and practical, illustrating exactly how an SMB (specify type if possible, e.g., 'a small marketing agency', 'a local restaurant') in the '{specific_smb_sector.strip() if specific_smb_sector else industry}' sector can implement the article's advice.
    7.  **No Extraneous Content:** Do NOT include any preamble, sign-off, mention of the context/outline, or any text other than the final article content formatted in Markdown. Do NOT include a "Scorecard" section.

    """

    # Add extra instructions if provided (for regeneration)
    if extra_instructions and extra_instructions.strip():
        prompt += f"\n**Additional Instructions for this Generation:**\n{extra_instructions.strip()}\n"

    prompt += "\nGenerate the complete article now, following all instructions meticulously."
    print("  ✅ Article prompt constructed.", flush=True)

    # --- API Call ---
    try:
        print(f"  ⏳ Sending request to AI API for article (Prompt length: {len(prompt)} chars)...", flush=True)
        # Use a slightly lower temperature for more focused writing based on the outline
        generation_config = genai.types.GenerationConfig(temperature=0.6)
        # Safety settings are defined with the model in Cell 3
        response = model.generate_content(prompt, generation_config=generation_config)
        print("  ⏳ Received article response from AI API.", flush=True)

        # --- Process Response ---
        if response.parts:
             generated_article = response.text
             print("    -> Status: ✅ OK (Article content received)", flush=True)
             if not generated_article or generated_article.strip() == "":
                 print("    -> Status: ⚠️ Warning (AI returned empty text for article)", flush=True)
                 return "⚠️ Warning: AI returned an empty article. Please try regenerating."
             else:
                 print(f"    -> Generated article length: {len(generated_article)}", flush=True)
                  # Basic cleanup: Remove potential leading/trailing ```markdown tags if present
                 generated_article = re.sub(r'^```markdown\s*', '', generated_article.strip())
                 generated_article = re.sub(r'\s*```$', '', generated_article)
                 return generated_article.strip()
        else:
             feedback = response.prompt_feedback
             block_reason = feedback.block_reason if feedback else "Unknown"
             safety_ratings_str = str(feedback.safety_ratings) if feedback else "N/A"
             print(f"    -> Status: ❌ ERROR (AI article response empty/blocked: {block_reason}) - Safety Ratings: {safety_ratings_str}", flush=True)
             error_message = f"❌ Error: AI article response was empty or blocked (Reason: {block_reason})."
             if feedback and feedback.safety_ratings:
                 error_message += f"\nSafety Concerns: {safety_ratings_str}"
             return error_message

    except Exception as e:
        print(f"  ❌ ERROR: Exception during AI API call for article:", flush=True)
        traceback.print_exc()
        error_details = str(e)
        if "API key not valid" in error_details:
            return "❌ Error: Invalid API Key. Please check configuration in Cell 3."
        elif "quota" in error_details.lower():
             return "❌ Error: API Quota Exceeded. Please check your usage limits."
        return f"❌ Error during article generation: {type(e).__name__} - {error_details}"
    finally:
        print(f"✅ [AI Article Generation] Function finished.", flush=True)

print("✅ AI article generation function defined.")

In [ ]:
# Cell 8: UI Helper Functions
# Defines functions used by the Gradio UI event handlers.

print("⏳ Defining UI helper functions...")

# --- Download Helpers (Remain the same) ---
def create_download_file(article_markdown, filename="generated_article.md"):
    """Creates a temporary Markdown file for download."""
    print("\n⏳ [create_download_file MD] Triggered...", flush=True)
    if not article_markdown or not isinstance(article_markdown, str) or article_markdown.strip() == "" or article_markdown.startswith("❌ Error:"):
        print("  ⚠️ Input empty/invalid for MD download. Aborted.", flush=True)
        # Return None or an appropriate Gradio update if needed
        return None # Can't return gr.update here directly if used only for file path

    temp_filepath = None
    try:
        # Use tempfile for secure temporary file creation
        with tempfile.NamedTemporaryFile(mode='w', suffix=".md", delete=False, encoding='utf-8') as temp_file:
            temp_file.write(article_markdown)
            temp_filepath = temp_file.name
        print(f"  ✅ Temp MD file created: {temp_filepath}", flush=True)
        return temp_filepath # Return the path for the DownloadButton
    except Exception as e:
        print(f"  ❌ ERROR creating MD download file:", flush=True)
        traceback.print_exc()
        # Clean up temp file if creation failed partially
        if temp_filepath and os.path.exists(temp_filepath):
             try: os.remove(temp_filepath)
             except OSError: pass
        return None # Indicate failure
    finally:
        print("✅ [create_download_file MD] Finished.", flush=True)

def create_txt_download_file(article_markdown, filename="generated_article.txt"):
    """Creates a temporary TXT file from Markdown for download."""
    print("\n⏳ [create_txt_download_file TXT] Triggered...", flush=True)
    if not article_markdown or not isinstance(article_markdown, str) or article_markdown.strip() == "" or article_markdown.startswith("❌ Error:"):
        print("  ⚠️ Input empty/invalid for TXT download. Aborted.", flush=True)
        return None

    plain_text = article_markdown # Default to original if conversion fails
    try:
        if markdownify:
            # Convert Markdown to plain text using markdownify
            plain_text = markdownify.markdownify(article_markdown, strip=['a', 'img', 'strong', 'em', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'code', 'pre', 'blockquote', 'hr', 'br'], heading_style="ATX").strip()
            # Basic cleanup after markdownify
            plain_text = re.sub(r'^\s*[\*\-\+]\s+', '', plain_text, flags=re.MULTILINE) # Remove list markers
            plain_text = re.sub(r'\n{3,}', '\n\n', plain_text) # Consolidate newlines
            print("  ℹ️ Converted MD to TXT via Markdownify.", flush=True)
        else:
             # Fallback conversion if markdownify is not available
             print("  ℹ️ Markdownify unavailable, using basic regex for TXT.", flush=True)
             text = re.sub(r'\*\*(.*?)\*\*|__(.*?)__', r'\1\2', article_markdown) # Bold
             text = re.sub(r'\*(.*?)\*|_(.*?)_', r'\1\2', text) # Italic
             text = re.sub(r'^#+\s+', '', text, flags=re.MULTILINE) # Headers
             text = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', text) # Links
             text = re.sub(r'^\s*[\*\-\+]\s+', '', text, flags=re.MULTILINE) # List items
             text = re.sub(r'`(.*?)`', r'\1', text) # Inline code
             text = re.sub(r'^```.*?^```', '', text, flags=re.MULTILINE | re.DOTALL) # Code blocks
             text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE) # Blockquotes
             text = re.sub(r'^---.*$', '', text, flags=re.MULTILINE) # Horizontal rules
             text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE) # <br> tags
             text = re.sub(r'\n{3,}', '\n\n', text) # Consolidate newlines
             plain_text = text.strip()

    except Exception as e:
        print(f"  ⚠️ WARN: Failed to convert MD to TXT: {e}. Using original Markdown.", flush=True)
        # Keep plain_text as the original markdown

    temp_filepath = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix=".txt", delete=False, encoding='utf-8') as temp_file:
            temp_file.write(plain_text)
            temp_filepath = temp_file.name
        print(f"  ✅ Temp TXT file created: {temp_filepath}", flush=True)
        return temp_filepath
    except Exception as e:
        print(f"  ❌ ERROR creating TXT download file:", flush=True)
        traceback.print_exc()
        if temp_filepath and os.path.exists(temp_filepath):
             try: os.remove(temp_filepath)
             except OSError: pass
        return None
    finally:
        print("✅ [create_txt_download_file TXT] Finished.", flush=True)


# --- Generation Result Parsing and UI Update Helper ---
def parse_generation_result(result):
    """
    Parses the AI generation result (outline or article).
    Determines if it's an error or success and prepares UI updates.

    Args:
        result (str): The raw output from the AI generation function.

    Returns:
        tuple: (str, str) - A tuple containing:
            - main_content (str): The processed content (outline or article) or error message.
            - status_message (str): A status message for the UI.
    """
    print("\n⏳ [parse_generation_result] Parsing AI result...", flush=True)
    print(f"  DEBUG: Result received: Type={type(result)}, Length={len(result) if isinstance(result, str) else 'N/A'}", flush=True)

    main_content = ""
    status_message = "✅ Generation successful."

    if isinstance(result, str):
        if result.startswith("❌ Error:") or result.startswith("⚠️ Warning:"):
            status_message = result # Use the error/warning as the status
            main_content = f"**Generation Failed/Warning:**\n{result}" # Display error in content area
            print(f"  🚦 Generation failed or has warning. Status: {status_message}", flush=True)
        elif result.strip() == "":
            status_message = "⚠️ Warning: Generation returned empty content."
            main_content = status_message
            print(f"  🚦 Generation successful but result is empty.", flush=True)
        else:
            main_content = result # Successful generation
            print(f"  🚦 Generation successful. Content length: {len(main_content)}", flush=True)
    else:
        status_message = "❌ Error: Unknown generation failure type."
        main_content = status_message
        print(f"  ❌ Unknown generation failure. Result type: {type(result)}", flush=True)

    print(f"✅ [parse_generation_result] UI update values prepared.", flush=True)
    # Return tuple: (content_to_display, status_message_for_ui)
    return (main_content, status_message)


# --- Button Interactivity Helpers ---
def disable_buttons_processing():
    """Disables buttons during source processing."""
    print("  UI: Disabling Process button.", flush=True)
    return gr.update(interactive=False) # Only disable process button

def enable_buttons_processing():
    """Enables buttons after source processing."""
    print("  UI: Enabling Process button.", flush=True)
    return gr.update(interactive=True)

def disable_buttons_outline_gen():
    """Disables buttons during outline generation."""
    print("  UI: Disabling Outline Gen buttons.", flush=True)
    return { # Return dict to update multiple components
        outline_generate_button: gr.update(interactive=False),
        outline_regenerate_button: gr.update(interactive=False),
        outline_status_output: gr.update(value="⏳ Generating outline...")
    }

def enable_buttons_outline_gen():
    """Enables buttons after outline generation."""
    print("  UI: Enabling Outline Gen buttons.", flush=True)
    return {
         outline_generate_button: gr.update(interactive=True),
         outline_regenerate_button: gr.update(interactive=True)
         # Status is updated separately by parse_generation_result
    }

def disable_buttons_article_gen():
    """Disables buttons during article generation."""
    print("  UI: Disabling Article Gen buttons.", flush=True)
    return { # Return dict to update multiple components
        generate_button: gr.update(interactive=False),
        regenerate_button: gr.update(interactive=False),
        generation_status_output: gr.update(value="⏳ Generating article...")
    }

def enable_buttons_article_gen():
    """Enables buttons after article generation."""
    print("  UI: Enabling Article Gen buttons.", flush=True)
    return {
        generate_button: gr.update(interactive=True),
        regenerate_button: gr.update(interactive=True)
        # Status is updated separately by parse_generation_result
    }

# --- Other UI Update Helpers ---
def update_sources_display(sources):
    """Updates the Markdown display for processed sources."""
    print(f"  ℹ️ [UI] Updating processed sources display.", flush=True)
    if isinstance(sources, list):
        if sources:
            return "* " + "\n* ".join(sources)
        else:
            return "*No sources processed successfully yet.*"
    else:
        print(f"  ⚠️ [UI] Warning: Expected list for sources, got {type(sources)}. Displaying default.", flush=True)
        return "*Error updating sources or no sources processed.*"

def switch_tab(tab_index):
    """Switches the active tab."""
    print(f"  ℹ️ [UI] Switching to Tab {tab_index + 1}.", flush=True)
    return gr.update(selected=tab_index)

def check_status_and_switch_tab(status, success_tab_index):
     """Switches tab only if status indicates success."""
     print(f"  ℹ️ [UI] Checking status to switch tab: {status[:50]}...", flush=True)
     if isinstance(status, str) and not status.strip().startswith("❌") and not status.strip().startswith("⚠️"):
         print(f"    -> Success status detected. Switching to Tab {success_tab_index + 1}.", flush=True)
         return gr.update(selected=success_tab_index)
     else:
         print("    -> Error/Warning status detected or invalid status type. Staying on current tab.", flush=True)
         return gr.update() # No change

print("✅ UI helper functions defined.")

# Define UI components globally IF they are needed in helper functions before the UI is built
# This is necessary for the enable/disable button helpers that return dictionaries
# We only need the *names* for the keys in the dictionary, the actual objects will be assigned later
outline_generate_button = None
outline_regenerate_button = None
outline_status_output = None
generate_button = None
regenerate_button = None
generation_status_output = None

In [ ]:
# Cell 9: Gradio UI Definition (Layout & Event Handlers)
# Defines the Gradio interface structure and connects UI elements to functions.
# Includes Outline Generation/Editing and Article Regeneration.

print("⏳ Defining Gradio UI layout and event handlers...")

# --- Gradio App Definition ---
inter_font = gr.themes.GoogleFont("Inter")
custom_css = """
<style>
    /* Increase base font size slightly */
    .gradio-container { font-size: 105%; }
    /* Word wrap for Markdown output */
    #article_output_markdown .gradio-markdown, #outline_output_markdown .gradio-markdown {
        word-wrap: break-word; white-space: pre-wrap; overflow-wrap: break-word;
    }
    /* Ensure Textbox for outline has reasonable height */
     #outline_edit_textbox textarea { min-height: 300px !important; }
</style>
"""

# Assuming necessary imports (gr, etc.) and constants (INDUSTRY_CHOICES, etc.)
# and functions (disable_buttons_processing, etc.) are available from previous cells.
# This code block focuses *only* on the UI definition with added emojis.

with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", font=inter_font), css=custom_css) as app:
    # App Title
    gr.Markdown("# Contextual Article Writer ✍️📄") # Added emojis
    gr.Markdown("Generate structured articles, starting with an outline.")

    # --- State variables ---
    # Store intermediate data between steps
    scraped_content_state = gr.State("")        # Holds raw text from sources
    scraping_status_state = gr.State("")        # Holds status message from scraping
    successful_sources_state = gr.State([])     # Holds list of successfully processed source names
    generated_outline_state = gr.State("")      # Holds the latest generated/edited outline
    generated_article_state = gr.State("")      # Stores the final generated article

    # --- UI Tabs Structure ---
    with gr.Tabs() as tabs:

        # --- Tab 1: Input Sources ---
        with gr.TabItem("1. Input Sources 📥", id=0): # Added emoji
            gr.Markdown("Enter URLs (one per line) and/or upload supported files (PDF, DOCX) to extract context.")
            with gr.Row():
                url_input = gr.Textbox(lines=5, placeholder="https://example.com/article1\nhttps://example.com/article2", label="Reference URLs 🔗", info="Enter one full URL per line.", scale=1) # Added emoji
                file_input = gr.Files(label="Upload Files (PDF/DOCX) 📄", file_types=[".pdf", ".docx"], scale=1) # Added emoji
            # Assign the button object to the global name
            process_button = gr.Button("Process Sources & Go to Outline ⚙️➡️", variant="primary") # Added emojis
            scraping_status_output_tab1 = gr.Textbox(label="Processing Status 📊", interactive=False, lines=5, max_lines=10, placeholder="Status will appear here after processing...") # Added emoji
            with gr.Accordion("Preview Extracted Content (Max 3000 chars) 👀", open=False): # Added emoji
                 scraped_content_preview = gr.Textbox(label="Extracted Content Preview", lines=8, interactive=False)

        # --- Tab 2: Generate & Edit Outline ---
        with gr.TabItem("2. Generate & Edit Outline 📝", id=1): # Added emoji
            gr.Markdown("Generate an article outline based on the processed content and your prompt. **You can edit the outline below before generating the final article.**")
            with gr.Accordion("Context Sources Used 📚", open=False): # Added emoji
                processed_sources_display_tab2 = gr.Markdown(value="*No sources processed yet.*")

            with gr.Row():
                user_prompt_input = gr.Textbox(label="Article Prompt / Topic ❓", placeholder="e.g., 'How can SMBs leverage AI for customer service?'", info="What should the article be about?", scale=2) # Added emoji

            # Outline Generation Parameters
            with gr.Row():
                # Assuming INDUSTRY_CHOICES is loaded from Cell 4
                industry_default = INDUSTRY_CHOICES[0] if INDUSTRY_CHOICES else None
                industry_input_outline = gr.Dropdown(INDUSTRY_CHOICES, label="Target Industry 🏭", value=industry_default, info="Select the primary industry focus.", scale=1) # Added emoji

                # Assuming TARGET_BUSINESS_CHOICES is loaded from Cell 4
                target_default = TARGET_BUSINESS_CHOICES[0] if TARGET_BUSINESS_CHOICES else None
                target_business_input_outline = gr.Dropdown(TARGET_BUSINESS_CHOICES, label="Target Business 🎯", value=target_default, info="Select the primary audience.", scale=1) # Added emoji
            specific_smb_sector_input_outline = gr.Textbox(label="Specific SMB Sector (if selected above)", placeholder="e.g., 'Local Restaurants', 'Independent Bookstores'", info="Required if 'Specific SMB Sector' is chosen.", interactive=True, visible=False) # Start hidden

            # Outline Generation Buttons
            with gr.Row():
                 # Assign button objects to global names
                 outline_generate_button = gr.Button("Generate Outline 💡", variant="primary", scale=1) # Added emoji
                 outline_regenerate_button = gr.Button("Regenerate Outline 🔄", variant="secondary", scale=1) # Added emoji

            # Outline Status & Output/Editor
            outline_status_output = gr.Textbox(label="Outline Generation Status 📊", interactive=False, placeholder="Status will appear here...") # Added emoji, Assign global name
            outline_output_edit_textbox = gr.Textbox(label="Generated Outline (Editable) ✏️", lines=15, interactive=True, elem_id="outline_edit_textbox", placeholder="Outline will appear here. You can edit it before generating the article.") # Added emoji, Editable textbox

            # Button to proceed
            confirm_outline_button = gr.Button("Confirm Outline & Go to Article Generation ✅➡️", variant="primary") # Added emojis


        # --- Tab 3: Configure & Generate Article ---
        with gr.TabItem("3. Generate Article ✨", id=2): # Added emoji
            gr.Markdown("Generate the full article based on the confirmed outline and additional settings.")
            with gr.Accordion("Confirmed Outline (Read-only) ✅", open=True): # Added emoji
                 # *** This line had the fix from previous steps ***
                 confirmed_outline_display = gr.Markdown(value="*Outline will be displayed here once confirmed.*")
            with gr.Accordion("Context Sources Used 📚", open=False): # Added emoji
                processed_sources_display_tab3 = gr.Markdown(value="*No sources processed yet.*")

            # Article Generation Parameters (some might be inherited or just displayed)
            with gr.Row():
                 # Assuming TONE_CHOICES is loaded from Cell 4
                 tone_default = TONE_CHOICES[0] if TONE_CHOICES else None
                 tone_input_article = gr.Dropdown(TONE_CHOICES, label="Article Tone 🎭", value=tone_default, info="Select the desired writing style.") # Added emoji
                 word_count_input_article = gr.Number(label="Approximate Word Count 🔢", value=750, minimum=100, step=50, info="AI will aim for this length.") # Added emoji

            # Regeneration Prompt
            extra_instructions_input = gr.Textbox(label="Optional: Extra Instructions for Regeneration 💬", placeholder="e.g., 'Focus more on cost savings', 'Make the introduction shorter'", info="Use this if you need to regenerate the article with specific changes.", lines=2) # Added emoji

            # Article Generation Buttons
            with gr.Row():
                 # Assign button objects to global names
                 generate_button = gr.Button("Generate Article ✨", variant="primary", scale=1) # Added emoji
                 regenerate_button = gr.Button("Regenerate Article 🔄📄", variant="secondary", scale=1) # Added emojis

            # Article Generation Status
            generation_status_output = gr.Textbox(label="Article Generation Status 📊", interactive=False, placeholder="Status will appear here...") # Added emoji, Assign global name


        # --- Tab 4: View & Download Article ---
        with gr.TabItem("4. View & Download Article 💾", id=3): # Added emoji
            gr.Markdown("Review the generated article below. Use the buttons to download the full content.")
            # REMOVED Scorecard Section

            # Main Article Content Display
            generated_article_output = gr.Markdown(label="Generated Article Content 📄", elem_id="article_output_markdown", value="*Article content will appear here after generation.*") # Added emoji

            # Download Buttons
            with gr.Row():
                download_button_md = gr.DownloadButton(label="Download Article (.md) 💾", variant="secondary", scale=1) # Added emoji
                download_button_txt = gr.DownloadButton(label="Download Article (.txt) 💾", variant="secondary", scale=1) # Added emoji


    # --- Helper function to update visibility of specific SMB sector input ---
    def update_smb_visibility(target_choice):
        # Assuming TARGET_BUSINESS_CHOICES is loaded and available
        if target_choice == "Specific SMB Sector (Defined Below)":
            return gr.update(visible=True)
        else:
            return gr.update(visible=False, value="") # Hide and clear value

    # Assuming target_business_input_outline is defined above
    target_business_input_outline.change(
        fn=update_smb_visibility,
        inputs=target_business_input_outline,
        outputs=specific_smb_sector_input_outline
    )
    # Initialize visibility based on default value
    # Assuming app is the gr.Blocks() instance
    app.load(fn=update_smb_visibility, inputs=target_business_input_outline, outputs=specific_smb_sector_input_outline)


    # --- Event Handlers ---
    # Assuming all functions (disable_buttons_processing, scrape_urls_and_files, etc.)
    # and components (process_button, tabs, etc.) are defined and available in scope.

    # 1. Process Sources Button (Tab 1)
    process_button.click(
        fn=disable_buttons_processing, # Disables process_button
        outputs=[process_button]
    ).then(
        fn=scrape_urls_and_files, # Runs scraping
        inputs=[url_input, file_input],
        outputs=[scraped_content_state, scraping_status_state, successful_sources_state] # Updates states
    ).then(
        # Update UI elements based on state changes
        fn=lambda status, content, sources: {
            scraping_status_output_tab1: status,
            scraped_content_preview: content[:3000] + ("..." if len(content) > 3000 else ""),
            processed_sources_display_tab2: update_sources_display(sources), # Update display in Tab 2
            processed_sources_display_tab3: update_sources_display(sources), # Update display in Tab 3
        },
        inputs=[scraping_status_state, scraped_content_state, successful_sources_state],
        outputs=[scraping_status_output_tab1, scraped_content_preview, processed_sources_display_tab2, processed_sources_display_tab3]
    ).then(
        fn=enable_buttons_processing, # Re-enables process_button
        outputs=[process_button]
    ).then(
        # Switch to Tab 2 (Outline Gen) only on success
        fn=check_status_and_switch_tab,
        inputs=[scraping_status_state, gr.State(1)], # Pass status and target tab index (1)
        outputs=[tabs]
    )

    # 2. Generate Outline Button (Tab 2)
    outline_generate_button.click(
        fn=disable_buttons_outline_gen, # Disables outline buttons, updates status
        outputs=[outline_generate_button, outline_regenerate_button, outline_status_output]
    ).then(
        fn=generate_outline_with_gemini, # Runs outline generation
        inputs=[scraped_content_state, user_prompt_input, industry_input_outline, target_business_input_outline, specific_smb_sector_input_outline],
        outputs=[generated_outline_state] # Stores result in state
    ).then(
        fn=parse_generation_result, # Parses result (content, status)
        inputs=[generated_outline_state],
        outputs=[outline_output_edit_textbox, outline_status_output] # Updates outline editor and status
    ).then(
        fn=enable_buttons_outline_gen, # Re-enables outline buttons
        outputs=[outline_generate_button, outline_regenerate_button]
    )

    # 3. Regenerate Outline Button (Tab 2) - Same logic as Generate
    outline_regenerate_button.click(
        fn=disable_buttons_outline_gen,
        outputs=[outline_generate_button, outline_regenerate_button, outline_status_output]
    ).then(
        fn=generate_outline_with_gemini,
        inputs=[scraped_content_state, user_prompt_input, industry_input_outline, target_business_input_outline, specific_smb_sector_input_outline],
        outputs=[generated_outline_state]
    ).then(
        fn=parse_generation_result,
        inputs=[generated_outline_state],
        outputs=[outline_output_edit_textbox, outline_status_output]
    ).then(
        fn=enable_buttons_outline_gen,
        outputs=[outline_generate_button, outline_regenerate_button]
    )

    # 4. Confirm Outline Button (Tab 2)
    confirm_outline_button.click(
        fn=lambda outline: { # Update state and display in Tab 3
            generated_outline_state: outline,
            confirmed_outline_display: gr.update(value=outline if outline else "*No outline confirmed yet.*")
        },
        inputs=[outline_output_edit_textbox], # Get edited outline
        outputs=[generated_outline_state, confirmed_outline_display] # Update state and Tab 3 display
    ).then(
        fn=switch_tab, # Switch to Tab 3 (Article Gen)
        inputs=[gr.State(2)], # Target tab index (2)
        outputs=[tabs]
    )

    # 5. Generate Article Button (Tab 3)
    generate_button.click(
        fn=disable_buttons_article_gen, # Disables article buttons, updates status
        outputs=[generate_button, regenerate_button, generation_status_output]
    ).then(
        fn=generate_article_with_gemini, # Runs article generation
        # Inputs: context, outline, industry, tone, target, word_count, specific_smb, extra_instructions=None
        inputs=[
            scraped_content_state,
            generated_outline_state, # Use the confirmed outline from state
            industry_input_outline, # Use industry from outline step
            tone_input_article,
            target_business_input_outline, # Use target from outline step
            word_count_input_article,
            specific_smb_sector_input_outline, # Use specific sector from outline step
            gr.State(None) # No extra instructions for initial generation
        ],
        outputs=[generated_article_state] # Stores result in state
    ).then(
        fn=parse_generation_result, # Parses result (content, status)
        inputs=[generated_article_state],
        outputs=[generated_article_output, generation_status_output] # Updates article display (Tab 4) and status (Tab 3)
    ).then(
        fn=enable_buttons_article_gen, # Re-enables article buttons
        outputs=[generate_button, regenerate_button]
    ).then(
        # Switch to Tab 4 (View Article) only on success
        fn=check_status_and_switch_tab,
        inputs=[generation_status_output, gr.State(3)], # Pass status and target tab index (3)
        outputs=[tabs]
    )

    # 6. Regenerate Article Button (Tab 3) - Similar to Generate, but includes extra instructions
    regenerate_button.click(
        fn=disable_buttons_article_gen,
        outputs=[generate_button, regenerate_button, generation_status_output]
    ).then(
        fn=generate_article_with_gemini,
         # Inputs: context, outline, industry, tone, target, word_count, specific_smb, extra_instructions
        inputs=[
            scraped_content_state,
            generated_outline_state,
            industry_input_outline,
            tone_input_article,
            target_business_input_outline,
            word_count_input_article,
            specific_smb_sector_input_outline,
            extra_instructions_input # Pass the extra instructions
        ],
        outputs=[generated_article_state]
    ).then(
        fn=parse_generation_result,
        inputs=[generated_article_state],
        outputs=[generated_article_output, generation_status_output]
    ).then(
        fn=enable_buttons_article_gen,
        outputs=[generate_button, regenerate_button]
    ).then(
        # Switch to Tab 4 (View Article) only on success
        fn=check_status_and_switch_tab,
        inputs=[generation_status_output, gr.State(3)], # Pass status and target tab index (3)
        outputs=[tabs]
    )

    # 7. Download Button Actions (Tab 4)
    # Use the generated_article_state which holds the full article content
    download_button_md.click(
        fn=create_download_file,
        inputs=[generated_article_state],
        outputs=[download_button_md] # Output is the file path for the button
    )
    download_button_txt.click(
        fn=create_txt_download_file,
        inputs=[generated_article_state],
        outputs=[download_button_txt] # Output is the file path for the button
    )

print("✅ Gradio UI defined successfully.")

In [ ]:
# Cell 10: Launch the Gradio App
# Runs the Gradio interface.

print("⏳ Launching Gradio App interface...")
print("   ℹ️ Detailed logs for processing and generation will appear here after UI interactions.")

# Assign UI components to global names *after* they are defined in the Blocks context
# This allows the enable/disable helper functions in Cell 8 to reference them correctly.
# We need to access the app's context or find the components by ID/reference if needed,
# but Gradio's event handler structure often handles this implicitly if components are in scope.
# For explicit updates via dictionaries (like enable/disable helpers), ensure the component
# objects are accessible. A simple way is to assign them within the `with app:` block.

# Example: If helpers needed direct access, you might need `app.blocks[...]` or pass components.
# However, the current structure where helpers return `gr.update` or dictionaries keyed by
# the component variables defined within `app` block scope should work.

if not api_key_configured or not model:
     print("\n" + "="*50)
     print("❌ CRITICAL ERROR: API Key not configured or AI Model failed to initialize.")
     print("   Please ensure your 'GOOGLE_API_KEY' is correctly set up in Kaggle Secrets (Add-ons -> Secrets).")
     print("   The application cannot function without a valid API key and model.")
     print("="*50 + "\n")
     # Optionally, display this message in the Gradio UI itself if possible,
     # maybe by updating a status label on load if the app instance is available globally.
     # However, launching might still proceed but generation will fail.
else:
    try:
        # share=True creates a public link (useful in Kaggle).
        # debug=False is usually preferred unless troubleshooting Gradio itself.
        app.launch(share=True, debug=False)
        print("\n✅ Gradio App launched. Click the public URL above to open the UI.")
        print("   NOTE: The first generation might take longer as the model 'warms up'.")
    except Exception as e:
        print(f"❌ [Cell 10] Failed to launch Gradio app: {e}")
        traceback.print_exc()

print("✅ Launch cell execution finished.")